In [23]:
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from datasets import load_dataset


sst2 = load_dataset('nyu-mll/glue', 'sst2')
sst2_train = sst2["train"]
sst2_val = sst2["validation"]


mrpc = load_dataset('nyu-mll/glue', 'mrpc')
mrpc_train = mrpc["train"]
mrpc_val = mrpc["validation"]

qqp = load_dataset("nyu-mll/glue", "qqp")
qqp_train = qqp["train"]
qqp_val = qqp["validation"]

mnli = load_dataset("nyu-mll/glue", "mnli")
mnli_train = mnli["train"]
mnli_val = mnli["validation_matched"]

In [24]:
import pandas as pd

sst2_train = pd.DataFrame(sst2_train)
sst2_val = pd.DataFrame(sst2_val)

mrpc_train = pd.DataFrame(mrpc_train)
mrpc_val = pd.DataFrame(mrpc_val)

qqp_train = pd.DataFrame(qqp_train)
qqp_val = pd.DataFrame(qqp_val)

mnli_train = pd.DataFrame(mnli_train)
mnli_val = pd.DataFrame(mnli_val)

In [25]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [26]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))

negation_words = {
    "no", "nor", "not",
    "don't", "didn't", "doesn't",
    "isn't", "aren't", "wasn't", "weren't",
    "haven't", "hasn't", "hadn't",
    "won't", "wouldn't", "can't", "couldn't",
    "shouldn't", "mustn't",
    "never"
}

stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

In [27]:
def preprocess(text):

    if pd.isna(text):
        return []

    # lowercase
    text = text.lower()

    # Expand contractions
    contractions = {
        "can't": "can not",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'ve": " have",
        "'ll": " will",
        "'d": " would",
        "'m": " am",
        "'s": " is"
    }

    for k, v in contractions.items():
        text = text.replace(k, v)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove HTML
    text = re.sub(r"<.*?>", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Keep only letters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    tokens = text.split()

    # Remove stopwords (except negation)
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # Lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    if len(tokens) == 0:
        tokens = ["<UNK>"]


    return tokens

In [28]:
sst2_train["sentence"] = sst2_train["sentence"].apply(preprocess)
sst2_val["sentence"] = sst2_val["sentence"].apply(preprocess)

mrpc_train["sentence1"] = mrpc_train["sentence1"].apply(preprocess)
mrpc_train["sentence2"] = mrpc_train["sentence2"].apply(preprocess)
mrpc_val["sentence1"] = mrpc_val["sentence1"].apply(preprocess)
mrpc_val["sentence2"] = mrpc_val["sentence2"].apply(preprocess)

qqp_train["question1"] = qqp_train["question1"].apply(preprocess)
qqp_train["question2"] = qqp_train["question2"].apply(preprocess)
qqp_val["question1"] = qqp_val["question1"].apply(preprocess)
qqp_val["question2"] = qqp_val["question2"].apply(preprocess)

mnli_train["premise"] = mnli_train["premise"].apply(preprocess)
mnli_train["hypothesis"] = mnli_train["hypothesis"].apply(preprocess)
mnli_val["premise"] = mnli_val["premise"].apply(preprocess)
mnli_val["hypothesis"] = mnli_val["hypothesis"].apply(preprocess)

In [29]:
# SST-2
sst2_train.dropna(inplace=True)
sst2_val.dropna(inplace=True)

# MRPC
mrpc_train.dropna(inplace=True)
mrpc_val.dropna(inplace=True)

# QQP
qqp_train.dropna(inplace=True)
qqp_val.dropna(inplace=True)

# MNLI
mnli_train.dropna(inplace=True)
mnli_val.dropna(inplace=True)

In [30]:
!pip install -q gensim

In [31]:
# ===========================
# Install & Import
# ===========================
import gensim.downloader as api
import numpy as np
import re

# ===========================
# Tokenize all datasets
# ===========================

print("Tokenizing datasets...")

sst2_tokens = sst2_train["sentence"].tolist()

mrpc_tokens = [
    s1 + ["<SEP>"] + s2
    for s1, s2 in zip(
        mrpc_train["sentence1"],
        mrpc_train["sentence2"]
    )
]

qqp_tokens = [
    q1 + ["<SEP>"] + q2
    for q1, q2 in zip(
        qqp_train["question1"],
        qqp_train["question2"]
    )
]

mnli_tokens = [
    p + ["<SEP>"] + h
    for p, h in zip(
        mnli_train["premise"],
        mnli_train["hypothesis"]
    )
]

all_tokens = (
    sst2_tokens
    + mrpc_tokens
    + qqp_tokens
    + mnli_tokens
)

# ===========================
# Load GloVe
# ===========================

print("Loading GloVe...")

glove_model = api.load("glove-wiki-gigaword-300")

print("Done!")

# ===========================
# Build Vocabulary
# ===========================

word2idx = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<SEP>": 2
}

counter = Counter(
    token
    for sent in all_tokens
    for token in sent
)

MIN_FREQ = 2

word2idx = {
    "<PAD>":0,
    "<UNK>":1,
    "<SEP>":2
}

for word, freq in counter.items():
    if freq >= MIN_FREQ:
        word2idx[word] = len(word2idx)

vocab_size = len(word2idx)
embed_dim = glove_model.vector_size

print("Vocabulary Size :", vocab_size)
print("Embedding Dimension :", embed_dim)

# ===========================
# Build Embedding Matrix
# ===========================

embedding_matrix = np.random.normal(
    0,
    0.1,
    (vocab_size, embed_dim)
)

embedding_matrix[0] = np.zeros(embed_dim)

covered = 0

for word, idx in word2idx.items():

    if word in glove_model:

        embedding_matrix[idx] = glove_model[word]

        covered += 1

print(f"Words Found in GloVe : {covered}")
print(f"Coverage : {covered/vocab_size:.2%}")

print("Embedding Matrix Shape :", embedding_matrix.shape)

Tokenizing datasets...
Loading GloVe...
Done!
Vocabulary Size : 74529
Embedding Dimension : 300
Words Found in GloVe : 60958
Coverage : 81.79%
Embedding Matrix Shape : (74529, 300)


In [32]:
MAX_LEN = 60

def text_to_indices(tokens):

    ids = [
        word2idx.get(tok, 1)
        for tok in tokens
    ]

    length = min(len(ids), MAX_LEN)

    if len(ids) < MAX_LEN:
        ids += [0] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids, length

In [33]:
class TextDataset(Dataset):

    def __init__(self, dataframe, text_columns, label_column="label"):

        self.samples = []

        for _, row in dataframe.iterrows():

            tokens = []

            for i, col in enumerate(text_columns):

                tokens += row[col]

                if i != len(text_columns) - 1:
                    tokens += ["<SEP>"]

            ids, length = text_to_indices(tokens)

            self.samples.append(
                (ids, length, row[label_column])
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        ids, length, label = self.samples[idx]

        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long)
        }

In [34]:
def create_dataloaders(train_df, val_df, text_columns, batch_size=128):

    train_dataset = TextDataset(train_df, text_columns)
    val_dataset = TextDataset(val_df, text_columns)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader

In [35]:
class BiRNNModel(nn.Module):

    def __init__(self, embedding_matrix, output_size):

        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(
                embedding_matrix,
                dtype=torch.float
            ),
            freeze=False,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_matrix.shape[1],
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(256, output_size)

    def forward(self, input_ids, lengths):

        x = self.embedding(input_ids)

        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.rnn(packed)

        h = torch.cat((hidden[-2], hidden[-1]), dim=1)

        h = self.dropout(h)

        return self.fc(h)

In [36]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
import os
def evaluate(model, loader, device):

    model.eval()

    preds = []
    labels = []

    start = time.perf_counter()

    with torch.no_grad():

        for batch in loader:

            logits = model(
                batch["input_ids"].to(device),
                batch["length"].to(device)
            )

            pred = logits.argmax(1)

            preds.extend(pred.cpu().numpy())
            labels.extend(batch["label"].numpy())

    inference_time = time.perf_counter() - start

    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro"),
        "f1": f1_score(labels, preds, average="macro"),
        "inference_time": inference_time
    }

    return metrics

In [37]:
import time
import os

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    model_name,
    epochs=10
):

    training_start = time.perf_counter()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=1e-2
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1
    )

    best_f1 = 0
    patience = 2
    counter = 0
    model_size = 0

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for batch in train_loader:

            optimizer.zero_grad()

            logits = model(
                batch["input_ids"].to(device),
                batch["length"].to(device)
            )

            loss = criterion(
                logits,
                batch["label"].to(device)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1
            )

            optimizer.step()

            total_loss += loss.item()

        metrics = evaluate(
            model,
            val_loader,
            device
        )

        scheduler.step(metrics["f1"])

        print(f"\nEpoch {epoch+1}")
        print(metrics)

        if metrics["f1"] > best_f1:

            best_f1 = metrics["f1"]
            counter = 0

            model_path = f"models/{model_name}.pt"

            torch.save(
                model.state_dict(),
                model_path
            )

            model_size = os.path.getsize(model_path) / (1024 * 1024)

            print(f"Model Size : {model_size:.2f} MB")

        else:

            counter += 1

            if counter >= patience:
                break

    training_time = time.perf_counter() - training_start

    print(f"\nTraining Time : {training_time:.2f} sec")
    print(f"Inference Time : {metrics['inference_time']:.4f} sec")
    print(f"Best F1 : {best_f1:.4f}")

    metrics["training_time"] = training_time
    metrics["model_size_mb"] = model_size

    return metrics

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [39]:
train_loader, val_loader = create_dataloaders(
    sst2_train,
    sst2_val,
    ["sentence"]
)

model = BiRNNModel(
    embedding_matrix,
    output_size=2
).to(device)

train_model(
    model,
    train_loader,
    val_loader,
    device,
    "best_sst2_model"
)


Epoch 1
{'accuracy': 0.7981651376146789, 'precision': 0.7981779884149552, 'recall': 0.797970868064326, 'f1': 0.7980365837610213, 'inference_time': 0.034188708000328916}
Model Size : 86.09 MB

Epoch 2
{'accuracy': 0.819954128440367, 'precision': 0.8200172748866336, 'recall': 0.819746148017176, 'f1': 0.8198287831230959, 'inference_time': 0.03267849199983175}
Model Size : 86.09 MB

Epoch 3
{'accuracy': 0.8119266055045872, 'precision': 0.8145548851487244, 'recall': 0.8127473267660184, 'f1': 0.8117592548716086, 'inference_time': 0.031834308999805216}

Epoch 4
{'accuracy': 0.8061926605504587, 'precision': 0.8070928172998123, 'recall': 0.8056432600825124, 'f1': 0.8058042097190139, 'inference_time': 0.03218508700001621}

Training Time : 45.42 sec
Inference Time : 0.0322 sec
Best F1 : 0.8198


{'accuracy': 0.8061926605504587,
 'precision': 0.8070928172998123,
 'recall': 0.8056432600825124,
 'f1': 0.8058042097190139,
 'inference_time': 0.03218508700001621,
 'training_time': 45.415914493999935,
 'model_size_mb': 86.09428119659424}

In [40]:
train_loader, val_loader = create_dataloaders(
    mrpc_train,
    mrpc_val,
    ["sentence1", "sentence2"]
)

model = BiRNNModel(
    embedding_matrix,
    output_size=2
).to(device)

train_model(
    model,
    train_loader,
    val_loader,
    device,
    "best_mrpc_model"
)


Epoch 1
{'accuracy': 0.7009803921568627, 'precision': 0.7033248081841432, 'recall': 0.5375510544302743, 'f1': 0.4911470047025148, 'inference_time': 0.03166538200002833}
Model Size : 86.09 MB

Epoch 2
{'accuracy': 0.6985294117647058, 'precision': 0.6692180876118701, 'recall': 0.5399266483287488, 'f1': 0.5002340287805607, 'inference_time': 0.036516589999791904}
Model Size : 86.09 MB

Epoch 3
{'accuracy': 0.6813725490196079, 'precision': 0.5945382530748384, 'recall': 0.5378011169459032, 'f1': 0.5127865961199295, 'inference_time': 0.022596441000132472}
Model Size : 86.09 MB

Epoch 4
{'accuracy': 0.6911764705882353, 'precision': 0.6281714785651793, 'recall': 0.5366341585396349, 'f1': 0.5006993006993007, 'inference_time': 0.020582530999945448}

Epoch 5
{'accuracy': 0.678921568627451, 'precision': 0.596185064935065, 'recall': 0.5526798366258231, 'f1': 0.5421424594166274, 'inference_time': 0.021298037000178738}
Model Size : 86.09 MB

Epoch 6
{'accuracy': 0.6642156862745098, 'precision': 0.579

{'accuracy': 0.6642156862745098,
 'precision': 0.575,
 'recall': 0.5481787113445028,
 'f1': 0.5416219053164184,
 'inference_time': 0.024809164000089368,
 'training_time': 6.840685699000005,
 'model_size_mb': 86.09428119659424}

In [41]:
train_loader, val_loader = create_dataloaders(
    qqp_train,
    qqp_val,
    ["question1", "question2"]
)

model = BiRNNModel(
    embedding_matrix,
    output_size=2
).to(device)

train_model(
    model,
    train_loader,
    val_loader,
    device,
    "best_qqp_model"
)


Epoch 1
{'accuracy': 0.7524610437793717, 'precision': 0.7388970201028591, 'recall': 0.7125196480908338, 'f1': 0.7202444642414525, 'inference_time': 1.7424406320001253}
Model Size : 86.09 MB

Epoch 2
{'accuracy': 0.7643334157803611, 'precision': 0.7500846337726342, 'recall': 0.7309000898320542, 'f1': 0.7375387550627707, 'inference_time': 1.6402632100002847}
Model Size : 86.09 MB

Epoch 3
{'accuracy': 0.7753400939896117, 'precision': 0.7604641047449987, 'recall': 0.7482590445322537, 'f1': 0.7531209824061667, 'inference_time': 1.6166202780000276}
Model Size : 86.09 MB

Epoch 4
{'accuracy': 0.7834281474152857, 'precision': 0.7709840688210028, 'recall': 0.753832478176623, 'f1': 0.7602370380205177, 'inference_time': 1.6170549299999948}
Model Size : 86.09 MB

Epoch 5
{'accuracy': 0.7855058125154588, 'precision': 0.7696443615374764, 'recall': 0.7678681597078877, 'f1': 0.7687265005480237, 'inference_time': 1.6217392850003307}
Model Size : 86.09 MB

Epoch 6
{'accuracy': 0.7857531535988128, 'pre

{'accuracy': 0.7939896116744991,
 'precision': 0.7799655015116644,
 'recall': 0.7722408695674472,
 'f1': 0.7756237700427112,
 'inference_time': 2.2924570350000977,
 'training_time': 611.0408680680002,
 'model_size_mb': 86.09427165985107}

In [42]:
train_loader, val_loader = create_dataloaders(
    mnli_train,
    mnli_val,
    ["premise", "hypothesis"]
)

model = BiRNNModel(
    embedding_matrix,
    output_size=3
).to(device)

train_model(
    model,
    train_loader,
    val_loader,
    device,
    "best_mnli_model"
)


Epoch 1
{'accuracy': 0.5286805909322465, 'precision': 0.535747845710881, 'recall': 0.5268117069452288, 'f1': 0.5290009401916874, 'inference_time': 0.47358959500024866}
Model Size : 86.10 MB

Epoch 2
{'accuracy': 0.5501782985226694, 'precision': 0.557010933336867, 'recall': 0.5459052955020454, 'f1': 0.5457351437597793, 'inference_time': 0.4539676790000158}
Model Size : 86.10 MB

Epoch 3
{'accuracy': 0.5590422822210902, 'precision': 0.5591527794091444, 'recall': 0.5595814064615232, 'f1': 0.5590827490663282, 'inference_time': 0.4718158229998153}
Model Size : 86.10 MB

Epoch 4
{'accuracy': 0.5660723382577687, 'precision': 0.5753201668546359, 'recall': 0.5650838772939178, 'f1': 0.5676878009488174, 'inference_time': 0.4571498979998978}
Model Size : 86.10 MB

Epoch 5
{'accuracy': 0.5792154865002547, 'precision': 0.5838080613853571, 'recall': 0.5760057735893084, 'f1': 0.5767679247147379, 'inference_time': 0.6715137149999464}
Model Size : 86.10 MB

Epoch 6
{'accuracy': 0.5744268976057055, 'pre

{'accuracy': 0.5761589403973509,
 'precision': 0.5787248988941861,
 'recall': 0.5736017031169992,
 'f1': 0.574446316511794,
 'inference_time': 0.46546004600031665,
 'training_time': 503.91608429200005,
 'model_size_mb': 86.09525775909424}